In [ ]:
# Experiment to test MLLS EM estimation of test priors using Ringnorm dataset

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torchhd.datasets import Ringnorm
import random

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# Parameters
train_size_per_class = 500
test_size = 1000
input_dim = 20
hidden_dim = 10
num_epochs = 100
learning_rate = 0.01
test_priors = [0.1 * i for i in range(1, 10)]  # [0.1, 0.2, ..., 0.9]
n_repeats = 10
max_em_iter = 5
tol = 1e-3

# Define simple MLP
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))
        return torch.sigmoid(self.fc2(x)).view(-1)

# MLLS EM estimation
def run_em(probs, train_pi, max_iter=20, tol=1e-3):
    new_pi = train_pi
    for i in range(max_iter):
        nom = (new_pi / train_pi) * probs
        denom = nom + ((1 - new_pi) / (1 - train_pi)) * (1 - probs)
        adjusted_post = nom / denom
        updated_pi = adjusted_post.mean().item()
        if abs(updated_pi - new_pi) < tol:
            break
        new_pi = updated_pi
    print(f"EM converged in {i+1} iterations, estimated π': {new_pi}")
    return new_pi

# Storage
results = {p: [] for p in test_priors}

# Main experiment
for _ in range(n_repeats):
    # Prepare training data with balanced priors (0.5 / 0.5)
    full_data = Ringnorm(root='./data', train=True, download=True)
    x_train = []
    y_train = []
    x_test, y_test = [], []
    pos_count, neg_count = 0, 0
    for x, y in full_data:
        if y == 1:
            if pos_count < train_size_per_class:
                x_train.append(x)
                y_train.append(1)
                pos_count += 1
            else:
                x_test.append(x)
                y_test.append(1)
        elif y == 0:
            if neg_count < train_size_per_class:
                x_train.append(x)
                y_train.append(0)
                neg_count += 1
            else:
                x_test.append(x)
                y_test.append(0)


    x_train = torch.stack(x_train)
    y_train = torch.tensor(y_train)

    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

    # Train model
    model = MLP(input_dim=input_dim, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        for xb, yb in train_loader:
            pred = model(xb)
            loss = F.binary_cross_entropy(pred, yb.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Run test sets for various priors
    full_test_data = list(zip(x_test, y_test))
    for pi_test in test_priors:
        print(f'\nTest prior π\'={pi_test}')
        n_pos = int(test_size * pi_test)
        n_neg = test_size - n_pos
        print(f'Sampling {n_pos} positive and {n_neg} negative instances.')
        pos_samples = [x for x, y in full_test_data if y == 1][:n_pos]
        neg_samples = [x for x, y in full_test_data if y == 0][:n_neg]
        x_test_sel = torch.stack(pos_samples + neg_samples)
        print(f'selected test set size: {x_test_sel.shape[0]}')
        print(f'selected data: {x_test_sel}')
        # y_test_sel = torch.tensor([1]*n_pos + [0]*n_neg)

        with torch.no_grad():
            probs = model(x_test_sel).cpu()
            print(probs)

        estimated_pi = run_em(probs, train_pi=0.5, max_iter=max_em_iter, tol=tol)
        print(estimated_pi)
        results[pi_test].append(estimated_pi)

import pandas as pd

# Summarize results
summary = {
    "True π'": [],
    "Estimated π' (mean)": [],
    "Estimated π' (std)": [],
}

for pi_test, estimates in results.items():
    summary["True π'"].append(round(pi_test, 2))
    summary["Estimated π' (mean)"].append(round(np.mean(estimates), 4))
    summary["Estimated π' (std)"].append(round(np.std(estimates), 4))

df_results = pd.DataFrame(summary)
df_results
